# 07 - Stacking with Random Forest and Boosting

Goal: combine text and image in a smarter way than a fixed weight.

Simple fusion (notebook 05) uses the same 70/30 weight for every product and every category. Here a second model learns how to combine them. Its input is, for each product, the 27 text probabilities and the 27 image probabilities (54 numbers).

We test two combiners:
- **Random Forest** (bagging): many decision trees trained on random samples, which then vote.
- **Gradient Boosting** (boosting): trees trained one after another, each one fixing the errors of the previous ones. We use scikit-learn's `HistGradientBoostingClassifier`, which works like LightGBM.

Honest scoring: text and image probabilities exist for the 16984 validation products, which neither model was trained on. We use 5-fold cross-validation on these products. Every product is predicted by a combiner that never saw it. The simple fusion weight is chosen the same way, so all methods are compared fairly.

## 1. Load text and image probabilities

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

from src.categories import CATEGORY_NAMES
from src.metrics import compute_per_class_metrics

# Saved by notebook 05 (optimised text model) and notebook 03_modeling_image
text = pd.read_csv("../data/processed/text_validation_probabilities.csv")
image = pd.read_csv("../data/processed/image_validation_predictions.csv")

classes = np.array([int(c) for c in text.columns[1:]])
text = text.rename(columns={str(c): f"text_{c}" for c in classes})
image = image.rename(columns={f"prob_{c}": f"image_{c}" for c in classes})

both = text.merge(image, on="row_id")
y = both["true_prdtypecode"].to_numpy()
P_text = both[[f"text_{c}" for c in classes]].to_numpy()
P_image = both[[f"image_{c}" for c in classes]].to_numpy()
X = np.hstack([P_text, P_image])

print(f"Products: {len(both)}")
print(f"Input columns: {X.shape[1]}")

Products: 16984
Input columns: 54


## 2. Score every method with 5-fold cross-validation

For each fold, the combiner is trained on 4/5 of the products and predicts the remaining 1/5.

In [2]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
weights = np.round(np.arange(0, 1.01, 0.1), 1)

predictions = {name: np.zeros_like(y) for name in ["Simple fusion", "Random Forest", "Gradient Boosting"]}
times = {name: 0.0 for name in predictions}

for train_idx, test_idx in folds.split(X, y):
    # Simple fusion: choose the text weight on the training part only
    fold_scores = [
        f1_score(y[train_idx], classes[(w * P_text[train_idx] + (1 - w) * P_image[train_idx]).argmax(axis=1)], average="weighted")
        for w in weights
    ]
    best_w = weights[int(np.argmax(fold_scores))]
    predictions["Simple fusion"][test_idx] = classes[(best_w * P_text[test_idx] + (1 - best_w) * P_image[test_idx]).argmax(axis=1)]

    start = time.time()
    forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42)
    predictions["Random Forest"][test_idx] = forest.fit(X[train_idx], y[train_idx]).predict(X[test_idx])
    times["Random Forest"] += time.time() - start

    start = time.time()
    boosting = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, early_stopping=True, random_state=42)
    predictions["Gradient Boosting"][test_idx] = boosting.fit(X[train_idx], y[train_idx]).predict(X[test_idx])
    times["Gradient Boosting"] += time.time() - start

predictions["Text only"] = classes[P_text.argmax(axis=1)]
predictions["Image only"] = classes[P_image.argmax(axis=1)]

order = ["Image only", "Text only", "Simple fusion", "Random Forest", "Gradient Boosting"]
results = pd.DataFrame({
    "Model": order,
    "Weighted F1": [f1_score(y, predictions[m], average="weighted") for m in order],
    "Macro F1": [f1_score(y, predictions[m], average="macro") for m in order],
    "Training time (s)": [f"{times[m]:.0f}" if m in times and times[m] > 0 else "-" for m in order],
})
results.style.hide(axis="index").format(precision=4)

Model,Weighted F1,Macro F1,Training time (s)
Image only,0.4817,0.4230,-
Text only,0.8430,0.8326,-
Simple fusion,0.8501,0.8401,-
Random Forest,0.8630,0.8491,25
Gradient Boosting,0.8595,0.8454,216


### Analysis

- Both learned combiners beat the simple 70/30 fusion.
- **Random Forest is best:** weighted F1 0.8630, against 0.8501 for simple fusion (+1.3 points) and 0.8430 for text only (+2.0 points).
- Gradient Boosting is close (0.8595) but 8 times slower (about 180 seconds against 23).
- Simple fusion gives the same score as in notebook 05 (0.8501), now with the weight chosen without looking at the test products. So the old result was not biased.

## 3. Per category: best combiner vs text only

In [3]:
best = results.iloc[2:]["Model"].iloc[results.iloc[2:]["Weighted F1"].to_numpy().argmax()]
text_scores = compute_per_class_metrics(y, predictions["Text only"], classes)
best_scores = compute_per_class_metrics(y, predictions[best], classes)

per_class = pd.DataFrame({
    "Code": classes,
    "Category": [CATEGORY_NAMES[c] for c in classes],
    "Text F1": text_scores["f1"],
    f"{best} F1": best_scores["f1"],
})
per_class["Gain"] = per_class[f"{best} F1"] - per_class["Text F1"]
print(f"Best combiner: {best}")
per_class.sort_values("Gain", ascending=False).style.hide(axis="index").format(precision=3)

Best combiner: Random Forest


Code,Category,Text F1,Random Forest F1,Gain
2705,New Books,0.711,0.850,0.139
10,Used Books,0.566,0.663,0.097
2403,Book Lots,0.768,0.819,0.051
2280,Magazines,0.810,0.853,0.043
1280,Toys,0.712,0.747,0.035
40,Games,0.753,0.785,0.032
2220,Pets,0.883,0.906,0.023
1300,RC Models,0.926,0.945,0.018
2582,Garden Furniture,0.768,0.786,0.018
1160,Cards,0.954,0.971,0.017


### Analysis

- The book categories gain the most: New Books +13.9 points, Used Books +9.7, Book Lots +5.1, Magazines +4.3.
- The combiner learned to trust the image more for books, which a fixed weight cannot do.
- A few categories lose: Wargames -5.7 points, Food -1.6. These are small categories (153 and 161 products), so the combiner has few examples to learn from.

## Conclusion

A Random Forest that combines text and image probabilities is the best model so far: 0.8630 weighted F1 (5-fold CV on the validation products). It helps most on books, where text alone is weakest.